# CardioIA — Fase 4 · Ir Além 1: Ética e governança em Visão Computacional

**Pergunta:** o detector de cardiomegalia funciona igualmente bem para diferentes grupos de pacientes?

Este notebook:

1. **identifica limitações do dataset**: desbalanceamento, representatividade e fatores de confusão;
2. **aplica métricas de fairness** (`fairlearn`) por **sexo**, **faixa etária** e **incidência do exame (PA × AP)**, com intervalos de confiança por *bootstrap*;
3. **testa uma mitigação**: pós-processamento com limiares por grupo (`ThresholdOptimizer`), ajustados na validação e avaliados no teste;
4. **discute implicações éticas** e práticas de governança.

Entradas: `data/raiox_processado/indice.csv` (notebook 01) e `modelos/predicoes_{validacao,teste}.csv` (notebook 02).

## 0. Configuração

In [ ]:
import os, sys, json
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/cardioia")
    DATA_DIR = BASE_DIR / "data" / "raiox_processado"
    MODEL_DIR = Path("/content/drive/MyDrive/CardioIA-Fase4/modelos")
else:
    BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DATA_DIR = BASE_DIR / "data" / "raiox_processado"
    MODEL_DIR = BASE_DIR / "modelos"
FIG_DIR = BASE_DIR / "docs" / "prints" / "fase4"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import recall_score, precision_score, roc_auc_score, accuracy_score, f1_score, balanced_accuracy_score
from fairlearn.metrics import (MetricFrame, selection_rate, false_positive_rate, true_positive_rate,
                               demographic_parity_difference, demographic_parity_ratio, equalized_odds_difference)
from fairlearn.postprocessing import ThresholdOptimizer
import fairlearn

SEED = 42
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("fairlearn", fairlearn.__version__)

indice = pd.read_csv(DATA_DIR / "indice.csv")
config = json.loads((MODEL_DIR / "config_modelos.json").read_text(encoding="utf-8"))
teste = pd.read_csv(MODEL_DIR / "predicoes_teste.csv")
val = pd.read_csv(MODEL_DIR / "predicoes_validacao.csv")
LIMIAR = config["modelos"]["resnet50_tl"]["limiar"]
ATRIBUTOS = {"sexo": "Sexo", "faixa_etaria": "Faixa etária", "incidencia": "Incidência (PA/AP)"}
indice["y"] = (indice.rotulo == "cardiomegalia").astype(int)
for d in (indice, teste, val):
    d["sexo"] = d.sexo.map({"M": "Masculino", "F": "Feminino"})
print(f"Modelo avaliado: ResNet50 (TL) · limiar {LIMIAR:.3f} · teste com {len(teste):,} imagens")

## 1. Limitações do dataset

### 1.1 Composição da coorte por grupo

Antes de olhar o modelo, olhamos os dados. Comparamos a composição de cada classe e a **prevalência de cardiomegalia** (taxa-base) em cada grupo. Se a taxa-base muda entre grupos, uma métrica como a paridade demográfica vai mostrar diferença mesmo com um modelo perfeito. Por isso ela precisa ser interpretada com cuidado.

In [ ]:
linhas = []
for col, nome in ATRIBUTOS.items():
    g = indice.groupby(col, observed=True)
    t = pd.DataFrame({"imagens": g.size(), "% da coorte": 100 * g.size() / len(indice),
                      "pacientes": g.paciente_id.nunique(), "prevalência cardiomegalia (%)": 100 * g.y.mean()})
    t.index = pd.MultiIndex.from_product([[nome], t.index.astype(str)])
    linhas.append(t)
composicao = pd.concat(linhas).round(1)
composicao

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, (col, nome) in zip(axes, ATRIBUTOS.items()):
    ct = pd.crosstab(indice[col], indice.rotulo, normalize="columns").mul(100)
    ct[["cardiomegalia", "normal"]].plot.bar(ax=ax, rot=0, color=["#c0392b", "#95a5a6"])
    ax.set_title(f"{nome}: % dentro de cada classe"); ax.set_xlabel(""); ax.set_ylabel("%")
    for c in ax.containers: ax.bar_label(c, fmt="%.0f", fontsize=7)
plt.tight_layout(); plt.savefig(FIG_DIR / "03_composicao_grupos.png", bbox_inches="tight"); plt.show()

### 1.2 Limitações identificadas

| Limitação | Evidência | Risco |
|---|---|---|
| **Desbalanceamento de classes** | Cardiomegalia é 2,5% do NIH completo. Na nossa coorte usamos 1:2 | Sem pesos de classe, o modelo tenderia a prever sempre "normal" |
| **Confusão por incidência (AP × PA)** | A proporção de exames AP é maior entre os positivos (tabela acima) | Na incidência AP o coração aparece **ampliado** e o exame indica paciente acamado. O modelo pode aprender "AP ⇒ cardiomegalia" (*shortcut*) |
| **Marcadores no filme** | Textos como "PORTABLE" e "AP" aparecem gravados nas imagens | Atalho visual, a mesma confusão acima pelo caminho do pixel |
| **Desbalanceamento de sexo entre classes** | As mulheres são maioria entre os positivos e minoria entre os negativos | O sexo vira um preditor espúrio, e a calibração pode diferir |
| **Sub-representação pediátrica** | A faixa 0–17 anos tem poucas imagens | Desempenho em crianças incerto. O índice cardiotorácico normal também difere em crianças |
| **Fonte única** | Todas as imagens vêm do NIH Clinical Center (EUA), entre 1992 e 2015 | Pode não generalizar para outros aparelhos, protocolos e para a população brasileira |
| **Rótulos por NLP** | Os rótulos foram extraídos dos laudos por mineração de texto (acurácia >90% declarada pelos autores; auditorias independentes acharam erros) | O modelo aprende também o ruído dos rótulos, e o teste herda esse ruído |
| **Atributos ausentes** | Não há raça/etnia, índice de massa corporal ou condição socioeconômica | **Não é possível auditar vieses nesses eixos**, e isso é uma limitação em si |
| **Vários exames por paciente** | Até dezenas de imagens da mesma pessoa | Tratado com o *split* por paciente (notebook 01), mas pacientes frequentes pesam mais |

In [ ]:
# Quantificando a confusão por incidência: a prevalência muda muito entre AP e PA?
ct = pd.crosstab(indice.incidencia, indice.rotulo, margins=True)
ct["% cardiomegalia"] = (100 * ct.cardiomegalia / ct.All).round(1)
ct

## 2. Métricas de fairness

Avaliamos o modelo **ResNet50 (TL)** no conjunto de teste, com o mesmo limiar escolhido na validação. As métricas por grupo são:

* **Taxa de seleção:** % de exames sinalizados como cardiomegalia. É a base da **paridade demográfica**.
* **TPR (sensibilidade):** % dos casos reais detectados. A igualdade entre grupos é a **igualdade de oportunidade**.
* **FPR:** % de normais sinalizados por engano. TPR e FPR iguais entre grupos formam as **chances equalizadas** (*equalized odds*).
* **Precisão** e **AUC** por grupo.

Em triagem médica, a métrica de fairness mais relevante costuma ser a **igualdade de oportunidade**: todo grupo deveria ter a mesma chance de ter a doença detectada. A paridade demográfica só faz sentido se a prevalência real for igual entre os grupos, o que não é o caso aqui.

In [ ]:
def metricas_grupo(df, col, score="p_resnet", limiar=LIMIAR):
    yhat = (df[score] >= limiar).astype(int)
    mf = MetricFrame(
        metrics={"n": lambda y, p: len(y), "prevalência": lambda y, p: y.mean(),
                 "taxa de seleção": selection_rate, "TPR (sensibilidade)": true_positive_rate,
                 "FPR": false_positive_rate, "precisão": lambda y, p: precision_score(y, p, zero_division=0),
                 "acurácia": accuracy_score},
        y_true=df.y, y_pred=yhat, sensitive_features=df[col].astype(str))
    auc = df.groupby(df[col].astype(str)).apply(lambda d: roc_auc_score(d.y, d[score]) if d.y.nunique() == 2 else np.nan)
    return mf, mf.by_group.assign(AUC=auc)

def resumo_disparidades(df, col, score="p_resnet", limiar=LIMIAR):
    yhat = (df[score] >= limiar).astype(int)
    mf, tab = metricas_grupo(df, col, score, limiar)
    s = df[col].astype(str)
    return {"atributo": ATRIBUTOS[col],
            "dif. paridade demográfica": demographic_parity_difference(df.y, yhat, sensitive_features=s),
            "razão paridade demográfica": demographic_parity_ratio(df.y, yhat, sensitive_features=s),
            "dif. igualdade de oportunidade (TPR)": tab["TPR (sensibilidade)"].max() - tab["TPR (sensibilidade)"].min(),
            "dif. FPR": tab["FPR"].max() - tab["FPR"].min(),
            "dif. chances equalizadas": equalized_odds_difference(df.y, yhat, sensitive_features=s),
            "dif. AUC": tab["AUC"].max() - tab["AUC"].min()}

tabelas = {}
for col, nome in ATRIBUTOS.items():
    _, tab = metricas_grupo(teste, col)
    tabelas[col] = tab
    print(f"\n=== {nome} ===")
    display(tab.style.format({c: "{:.3f}" for c in tab.columns if c != "n"}).format({"n": "{:.0f}"}))

In [ ]:
disp = pd.DataFrame([resumo_disparidades(teste, c) for c in ATRIBUTOS]).set_index("atributo")
disp.to_csv(MODEL_DIR / "fairness_disparidades_resnet.csv")
disp.style.format("{:.3f}").background_gradient(cmap="Reds", subset=[c for c in disp.columns if c.startswith("dif")], vmin=0, vmax=.3)

### 2.1 As diferenças são estatisticamente relevantes?

O teste tem cerca de 1.250 imagens e alguns grupos são pequenos, então uma diferença de alguns pontos pode ser só ruído. Usamos **bootstrap por paciente** (1.000 reamostragens, sorteando pacientes e não imagens) para estimar o intervalo de 95% da diferença de TPR e de FPR entre os grupos. Se o intervalo não inclui zero, a diferença é consistente.

In [ ]:
def bootstrap_gap(df, col, g1, g2, metrica, n=1000, score="p_resnet", limiar=LIMIAR):
    rng = np.random.default_rng(SEED)
    pacientes = df.paciente_id.unique()
    por_pac = {p: i for p, i in df.groupby("paciente_id").indices.items()}
    gaps = []
    for _ in range(n):
        idx = np.concatenate([por_pac[p] for p in rng.choice(pacientes, len(pacientes), replace=True)])
        d = df.iloc[idx]
        vals = []
        for g in (g1, g2):
            dg = d[d[col].astype(str) == g]
            yhat = dg[score] >= limiar
            alvo = dg.y == (1 if metrica == "TPR" else 0)
            vals.append(yhat[alvo].mean() if alvo.any() else np.nan)
        gaps.append(vals[0] - vals[1])
    gaps = np.array(gaps)
    return np.nanmean(gaps), *np.nanpercentile(gaps, [2.5, 97.5])

comparacoes = [("sexo", "Feminino", "Masculino"), ("incidencia", "AP", "PA"),
               ("faixa_etaria", "60+", "18-39"), ("faixa_etaria", "40-59", "18-39")]
linhas = []
for col, g1, g2 in comparacoes:
    for met in ("TPR", "FPR"):
        m, lo, hi = bootstrap_gap(teste, col, g1, g2, met)
        linhas.append({"comparação": f"{g1} − {g2}", "métrica": met, "diferença": m, "IC95% inf": lo, "IC95% sup": hi,
                       "consistente?": "sim" if (lo > 0 or hi < 0) else "não (inclui 0)"})
ic = pd.DataFrame(linhas)
ic.style.format({"diferença": "{:+.3f}", "IC95% inf": "{:+.3f}", "IC95% sup": "{:+.3f}"})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (col, nome) in zip(axes, ATRIBUTOS.items()):
    t = tabelas[col][["TPR (sensibilidade)", "FPR", "precisão", "AUC"]]
    t.plot.bar(ax=ax, rot=0, color=["#2471a3", "#e67e22", "#7d3c98", "#1e8449"], width=.8)
    ax.set_ylim(0, 1.1); ax.set_title(f"ResNet50 por {nome.lower()}"); ax.set_xlabel("")
    for c in ax.containers: ax.bar_label(c, fmt="%.2f", fontsize=6, rotation=90, padding=2)
    ax.legend(fontsize=7, loc="upper right", ncol=2)
plt.tight_layout(); plt.savefig(FIG_DIR / "03_metricas_por_grupo.png", bbox_inches="tight"); plt.show()

### 2.2 Comparação com a CNN do zero

Será que o viés vem dos dados ou da arquitetura? Repetimos o resumo de disparidades para a CNN treinada do zero, com o limiar dela.

In [ ]:
lim_cnn = config["modelos"]["cnn_do_zero"]["limiar"]
comp = pd.concat({
    "ResNet50 (TL)": disp,
    "CNN do zero": pd.DataFrame([resumo_disparidades(teste, c, "p_cnn", lim_cnn) for c in ATRIBUTOS]).set_index("atributo"),
}, names=["modelo"])
comp[["dif. igualdade de oportunidade (TPR)", "dif. FPR", "dif. chances equalizadas", "dif. AUC"]].unstack(0).round(3)

## 3. Mitigação: limiares por grupo (pós-processamento)

O `ThresholdOptimizer` do fairlearn escolhe **um limiar para cada grupo** de forma a satisfazer uma restrição de fairness, maximizando um objetivo de desempenho. Aplicamos a restrição de **igualdade de TPR** (*true positive rate parity*), com objetivo de acurácia balanceada.

* **Ajuste na validação** e **avaliação no teste**, para não usar o teste em nenhuma decisão.
* A mitigação é aplicada primeiro por **sexo** (atributo protegido) e depois por **incidência**. No caso da incidência há uma justificativa clínica: o coração aparece ampliado em AP, e radiologistas já leem exames AP com critério diferente.

**Cuidado ético:** limiares por sexo exigem usar o sexo do paciente na hora da inferência. Isso é aceitável em contexto clínico, onde o dado já existe no prontuário, mas precisa ser documentado e consentido (LGPD, art. 11: dado sensível de saúde).

In [ ]:
class ModeloDeScores(BaseEstimator, ClassifierMixin):
    # Adaptador: expõe as probabilidades já calculadas pelo notebook 02 como um 'classificador' do sklearn.
    def fit(self, X, y=None):
        self.classes_ = np.array([0, 1]); self.ajustado_ = True; return self
    def predict(self, X):
        return np.asarray(X)[:, 0]

def mitigar(col):
    to = ThresholdOptimizer(estimator=ModeloDeScores().fit(None), constraints="true_positive_rate_parity",
                            objective="balanced_accuracy_score", prefit=True, predict_method="predict")
    to.fit(val[["p_resnet"]], val.y, sensitive_features=val[col].astype(str))
    yhat = to.predict(teste[["p_resnet"]], sensitive_features=teste[col].astype(str), random_state=SEED)
    return to, yhat

def avaliar(yhat, col):
    s = teste[col].astype(str)
    mf = MetricFrame(metrics={"TPR": true_positive_rate, "FPR": false_positive_rate}, y_true=teste.y, y_pred=yhat, sensitive_features=s)
    return {"acurácia": accuracy_score(teste.y, yhat), "acurácia balanceada": balanced_accuracy_score(teste.y, yhat),
            "recall": recall_score(teste.y, yhat), "precisão": precision_score(teste.y, yhat), "F1": f1_score(teste.y, yhat),
            "dif. TPR": mf.difference()["TPR"], "dif. FPR": mf.difference()["FPR"],
            "dif. chances equalizadas": equalized_odds_difference(teste.y, yhat, sensitive_features=s)}

yhat_base = (teste.p_resnet >= LIMIAR).astype(int)
resultado_mitigacao, otimizadores = {}, {}
for col in ("sexo", "incidencia"):
    to, yhat_mit = mitigar(col)
    otimizadores[col] = to
    resultado_mitigacao[(ATRIBUTOS[col], "antes (limiar único)")] = avaliar(yhat_base, col)
    resultado_mitigacao[(ATRIBUTOS[col], "depois (limiar por grupo)")] = avaliar(yhat_mit, col)
mitigacao = pd.DataFrame(resultado_mitigacao).T
mitigacao.to_csv(MODEL_DIR / "fairness_mitigacao.csv")
mitigacao.style.format("{:.3f}").background_gradient(cmap="Reds", subset=["dif. TPR", "dif. FPR", "dif. chances equalizadas"], vmin=0, vmax=.3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ("sexo", "incidencia")):
    _, yhat_mit = mitigar(col)
    grupos = sorted(teste[col].astype(str).unique())
    antes = [((yhat_base == 1) & (teste.y == 1) & (teste[col].astype(str) == g)).sum() / ((teste.y == 1) & (teste[col].astype(str) == g)).sum() for g in grupos]
    depois = [((yhat_mit == 1) & (teste.y == 1) & (teste[col].astype(str) == g)).sum() / ((teste.y == 1) & (teste[col].astype(str) == g)).sum() for g in grupos]
    x = np.arange(len(grupos)); w = .38
    b1 = ax.bar(x - w/2, antes, w, label="antes (limiar único)", color="#aab7b8")
    b2 = ax.bar(x + w/2, depois, w, label="depois (limiar por grupo)", color="#2471a3")
    ax.bar_label(b1, fmt="%.2f", fontsize=8); ax.bar_label(b2, fmt="%.2f", fontsize=8)
    ax.set_xticks(x, grupos); ax.set_ylim(0, 1.1); ax.set_ylabel("TPR (sensibilidade)")
    ax.set_title(f"Sensibilidade por {ATRIBUTOS[col].lower()} — teste"); ax.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR / "03_mitigacao.png", bbox_inches="tight"); plt.show()

## 4. Implicações éticas e práticas de governança

**O que os números dizem.** As tabelas acima mostram onde o modelo tem desempenho desigual. As diferenças cujo intervalo de confiança não inclui zero são vieses consistentes, que não se explicam por ruído amostral. As demais precisam de mais dados para serem confirmadas ou descartadas.

**Por que isso importa.** Um falso negativo em cardiomegalia atrasa a investigação de insuficiência cardíaca. Se a sensibilidade for menor em um grupo, esse grupo recebe sistematicamente um cuidado pior. É um **dano alocativo**, mesmo que a acurácia geral pareça boa.

**Práticas de mitigação propostas para o CardioIA**

| Etapa | Prática |
|---|---|
| Dados | Coletar exames de hospitais brasileiros e de diferentes fabricantes. Registrar raça/cor (padrão IBGE) com consentimento para viabilizar auditoria. Amostrar de forma estratificada por sexo, faixa etária e incidência. Auditar uma amostra de rótulos com radiologistas |
| Treino | Reponderar a perda por grupo × classe. Recortar ou mascarar as bordas e os marcadores de texto para reduzir atalhos. Validar com Grad-CAM se o foco está na silhueta cardíaca |
| Pós-processamento | Limiares por incidência (justificativa clínica) ou por grupo protegido, sempre ajustados em validação e documentados |
| Avaliação | Relatar métricas **por subgrupo** em todo novo modelo, com IC 95%. Definir um teto aceitável de disparidade de TPR antes do deploy (ex.: ≤ 5 p.p.) |
| Uso | Ferramenta de **apoio**, nunca de decisão autônoma. Médico no circuito. Exibir confiança e Grad-CAM ao usuário. Aviso claro no app |
| Governança | *Model card* público. Monitorar drift e fairness em produção. Seguir a LGPD (dado sensível, finalidade, minimização) e a RDC ANVISA 657/2022 (software como dispositivo médico) caso o produto saia do ambiente acadêmico |

O relatório resumido desta análise está em `docs/relatorios/relatorio_ir_alem1_etica_fairness.md`.